### Grid search to choose best params for the 23 protein model (ISS >24)


In [1]:
### Small grid search to choose the best params for the proteins search for ISS >24 (so 25 will be in the critical group)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import xgboost as xgb
import matplotlib.patches as mpatches
from matplotlib.font_manager import FontProperties
import itertools
from sklearn.model_selection import train_test_split
%config Completer.use_jedi = False
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score
from sklearn.metrics import plot_roc_curve
from sklearn.metrics import auc
from sklearn.metrics import precision_recall_fscore_support
from sklearn.model_selection import GridSearchCV

In [2]:
clin = pd.read_csv('clin_ready.csv', header = 0, index_col=0)
prot = pd.read_csv('raw_data_proteins_filtered.csv',header = 0, index_col = 0)
prot_list = pd.read_csv('prot_filtered.csv', header = 0, index_col=0)

prot = prot.filter(prot_list.iloc[:,0], axis = 1)
prot = prot.drop(labels = [1578], axis = 0)
clin = clin.drop(labels = [1578], axis = 0)
print(len(prot_list))
prot.shape

4979


(414, 4979)

In [3]:
prot_23 = ['PROC.3758.68', 'HIST1H1C.2987.37', 'ASB9.19601.15',
       'CPLX2.15321.8', 'HIST2H2BE.14143.8', 'AFM.4763.31', 'PTH.5954.62',
       'SERPINA7.2706.69', 'RAP2A.9885.41', 'ADSSL1.13998.26',
       'FCN2.13717.15', 'ADSL.5023.23', 'CLIC5.12475.48',
       'ALDH2.18381.16', 'PLTP.15475.4', 'CLEC1B.4332.6', 'TGM3.4471.50',
       'CCL22.3508.78', 'IL1RL1.4234.8', 'UBA2.12500.88',
       'SERPINF1.9211.19', 'IL19.3035.80', 'PDLIM4.12387.7']

y = clin['iss']

PROT_23 =prot.filter(items = prot_23) 

# split in train and test and then further split test in 2 (final result will be 70, 15, 15)
X_train, X_test_large, y_train, y_test_large = train_test_split(PROT_23, y, shuffle = True, test_size = 0.3, random_state = 2)   #132     
X_test, X_val, y_test, y_val = train_test_split(X_test_large, y_test_large, shuffle = True, test_size = 0.5, random_state = 2)        

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("X_val:", X_val.shape)

X_train: (289, 23)
X_test: (62, 23)
X_val: (63, 23)


In [4]:
#concatenate train and test 
train_test = X_train.append(X_test)
assert len(train_test) == (len(X_train) + len(X_test)), "Shape error - X"

y_train_test = y_train.append(y_test)
assert len(y_train_test) == (len(y_train) + len(y_test)), "Shape error - y"

/var/folders/qj/tdp1dq2138jdytx50m9g6p180000gr/T/ipykernel_34900/1272753432.py:2: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  train_test = X_train.append(X_test)
/var/folders/qj/tdp1dq2138jdytx50m9g6p180000gr/T/ipykernel_34900/1272753432.py:5: FutureWarning: The series.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  y_train_test = y_train.append(y_test)


In [5]:
depths = [2, 3, 4]
etas = [0.1, 0.2, 0.5]
y_train_test_ = [1 if i > 24 else 0 for i in y_train_test]

dtrain= xgb.DMatrix(train_test, label= y_train_test_)


for max_depth in depths:
    for eta in etas:
        
        params = {"eta":eta, 
                 "max_depth": max_depth}
        
        cv_results = xgb.cv(
            params,
            dtrain,
            seed=3,
            nfold=5,
            metrics={'auc'},
            early_stopping_rounds=35
        )
        
        #print(cv_results)
        best_rounds = np.argmax(cv_results['test-auc-mean'])
        best_test = cv_results['test-auc-mean'][best_rounds]
        best_round_train = np.argmax(cv_results['train-auc-mean'])
        best_train = cv_results['train-auc-mean'][best_round_train] 
        current_depth = max_depth
        current_eta = eta
        print("best_result", best_test, "best_round:",best_rounds,"best training:",
              best_train, "depth:", current_depth, "best eta:", current_eta)

#depth of 2 and eta of 0.2 seem good

best_result 0.8715293849828374 best_round: 8 best training: 0.9506890702126862 depth: 2 best eta: 0.1
best_result 0.8788763918198133 best_round: 9 best training: 0.9656785511682606 depth: 2 best eta: 0.2
best_result 0.8572866989838717 best_round: 5 best training: 0.9873926063890227 depth: 2 best eta: 0.5
best_result 0.8681200888328856 best_round: 8 best training: 0.9859562408426571 depth: 3 best eta: 0.1
best_result 0.8683870778171168 best_round: 7 best training: 0.9947994404267255 depth: 3 best eta: 0.2
best_result 0.8534068947170438 best_round: 6 best training: 0.999520407639063 depth: 3 best eta: 0.5
best_result 0.8503168280194828 best_round: 9 best training: 0.9985829470694952 depth: 4 best eta: 0.1
best_result 0.8584674507923182 best_round: 5 best training: 0.9998059741936155 depth: 4 best eta: 0.2
best_result 0.8330579251919469 best_round: 5 best training: 1.0 depth: 4 best eta: 0.5


In [6]:
# test on the val set the performance
y_val_=[1 if i >24 else 0 for i in y_val]

pos_weight = np.bincount(y_train_test_)[0]/np.bincount(y_val_)[1]
dtrain = xgb.DMatrix(train_test, label= y_train_test_)
dtest = xgb.DMatrix(X_val, label=y_val_)
param = {'max_depth': 2, 'eta': 0.2, 'objective': 'binary:logistic', 'scale_pos_weight':pos_weight}
param['eval_metric'] = 'auc'
num_round = 20 
evallist = [(dtest, 'eval'), (dtrain, 'train')]
progress = {}
bst = xgb.train(param, dtrain, num_round, evallist, evals_result = progress, verbose_eval=5, early_stopping_rounds=25)     

[0]	eval-auc:0.77174	train-auc:0.77191
[5]	eval-auc:0.87020	train-auc:0.94577
[10]	eval-auc:0.87468	train-auc:0.95888
[15]	eval-auc:0.88235	train-auc:0.96471
[19]	eval-auc:0.89578	train-auc:0.97071


/usr/local/anaconda3/envs/ipykernel_py38/lib/python3.8/site-packages/xgboost/core.py:525: FutureWarning: Pass `evals` as keyword args.  Passing these as positional arguments will be considered as error in future releases.
  warnings.warn(


In [ ]:
#worth mentioning that they are all very similar in terms of performance so there's isn't a big reson why to choose one instead of another 